In [ ]:
import pandas as pd
from tab2img.converter import Tab2Img
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import numpy as np
import random
import os
from PIL import Image


#### tab2img

In [ ]:
def gen_image(filePath, folder):
    selected_columns = ['Flow Duration', 'Total Fwd Packets',
        'Total Backward Packets', 'Fwd Packets Length Total',
        'Bwd Packets Length Total', 'Fwd Packet Length Max',
        'Fwd Packet Length Min', 'Fwd Packet Length Mean',
        'Fwd Packet Length Std', 'Bwd Packet Length Max',
        'Bwd Packet Length Min', 'Bwd Packet Length Mean',
        'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
        'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
        'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
        'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
        'Bwd IAT Max', 'Bwd IAT Min',  'Fwd Header Length',
        'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
        'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
        'Packet Length Std', 'Packet Length Variance', 
        'Down/Up Ratio',
        'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
        'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
            'Subflow Fwd Bytes', 'Subflow Bwd Packets',
        'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
        'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
        'Active Max', 'Active Min', 'Label']

    df_to_convert = pd.read_csv(filePath)
    df_to_convert = df_to_convert[selected_columns]
    len(selected_columns)

    X = df_to_convert.drop("Label", axis=1)
    y = df_to_convert["Label"]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # train (0.9), val (0.05), test (0.05)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y_encoded, test_size=0.05, random_state=42, stratify=y_encoded)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.05264, random_state=42, stratify=y_temp)

    unique_labels = pd.unique(y)
    unique_labels_encoded = pd.unique(y_encoded)
    print(unique_labels)
    print(unique_labels_encoded)

    tab2Img = Tab2Img()
    train_images = tab2Img.fit_transform(X_train.to_numpy(), y_train)
    joblib.dump(tab2Img, "models/tab2img.pkl")

    tab2Img = Tab2Img()

    train_images = tab2Img.fit_transform(X_train.to_numpy(), y_train)
    val_images = tab2Img.transform(X_val.to_numpy())
    test_images = tab2Img.transform(X_test.to_numpy())

    def save_images_by_label(images, labels, output_dir, img_size=224, prefix='img'):
        os.makedirs(output_dir, exist_ok=True)
        org_size = 8  # padding 8x8
        scale_factor = img_size // org_size

        # Group indices by label
        label_to_indices = {}
        for i, label in enumerate(labels):
            label_to_indices.setdefault(label, []).append(i)

        for label, indices in label_to_indices.items():
            os.makedirs(os.path.join(output_dir, str(label)), exist_ok=True)
            random.shuffle(indices)
            total = 200
            current_total = 0

            for count, i in enumerate(indices):
                img_array = images[i].reshape(org_size, org_size)
                if img_array.max() <= 1.0:  # Assuming data is normalized
                    img_array = (img_array * 255).astype(np.uint8)
                else:
                    img_array = img_array.astype(np.uint8)
                
                zoomed_array = np.kron(img_array, np.ones((scale_factor, scale_factor)))
                scaled_image = zoomed_array.astype(np.uint8)

                img = Image.fromarray(scaled_image, mode='L').convert("RGB")

                if random.random() < 0.8:  # 80% augmentation chance
                    img_array_aug = np.array(img)
                    
                    if random.random() < 0.3:
                        dropout_mask = np.random.random(img_array_aug.shape[:2]) > 0.9
                        img_array_aug[dropout_mask] = 0
                    
                    if random.random() < 0.4:
                        shift = random.randint(-5, 5)
                        img_array_aug = np.roll(img_array_aug, shift, axis=1)
                        if shift > 0:
                            img_array_aug[:, :shift] = 0
                        else:
                            img_array_aug[:, shift:] = 0
                    
                    if random.random() < 0.5:
                        img_array_aug = np.clip(img_array_aug * random.uniform(0.7, 1.3), 0, 255)
                    
                    img = Image.fromarray(img_array_aug.astype(np.uint8))

                img.save(os.path.join(output_dir, str(label), f'{prefix}_{count}.png'))
                if current_total >= total:
                    break

    # save train img
    save_images_by_label(train_images, y_train, output_dir=f'{folder}/train_images', prefix='train')
    # save val img
    save_images_by_label(val_images, y_val, output_dir=f'{folder}/val_images', prefix='val')
    # save test img
    save_images_by_label(test_images, y_test, output_dir=f'{folder}/test_images', prefix='test')

gen_image('csv/cicddos_2019_4_labels.csv', 'images')

['Syn' 'Group1' 'Group2' 'BENIGN']
[3 1 2 0]


f:\NCKH\code\train_model_2\.venv\Lib\site-packages\numpy\_core\_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
f:\NCKH\code\train_model_2\.venv\Lib\site-packages\tab2img\converter.py:24: RuntimeWarning: invalid value encountered in subtract
  cov  = (X - mean_X) * (Y - mean_Y).reshape(n_sample, 1)
f:\NCKH\code\train_model_2\.venv\Lib\site-packages\numpy\_core\_methods.py:52: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
C:\Users\NewTun\AppData\Local\Temp\ipykernel_20488\4228492224.py:79: RuntimeWarning: invalid value encountered in cast
  img_array = img_array.astype(np.uint8)


## convert img to folder

In [ ]:
def gen_image_into_a_folder(filePath, folder):
    selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                        'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                        'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                        'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                        'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                        'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                        'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                        'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                        'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                        'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                        'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

    df_to_convert = pd.read_csv(filePath)
    df_to_convert = df_to_convert[selected_columns]
    X = df_to_convert.drop("Label", axis=1)
    y = df_to_convert["Label"]
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    tab2Img = joblib.load("tab2img.pkl")
    all_images = tab2Img.transform(X.to_numpy())
    all_labels = y_encoded

    def save_images_by_label(images, labels, output_dir, img_size=224, prefix='img'):
        os.makedirs(output_dir, exist_ok=True)
        org_size = 7
        scale_factor = img_size // org_size

        total = len(images)  # Lưu toàn bộ ảnh
        current_total = 0

        for i, label in enumerate(labels):
            img_array = images[i].reshape(org_size, org_size)

            if img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)

            zoomed_array = np.kron(img_array, np.ones((scale_factor, scale_factor)))
            scaled_image = zoomed_array.astype(np.uint8)
            img = Image.fromarray(scaled_image, mode='L').convert("RGB")

            if random.random() < 0.8:
                img_array_aug = np.array(img)

                if random.random() < 0.3:
                    dropout_mask = np.random.random(img_array_aug.shape[:2]) > 0.9
                    img_array_aug[dropout_mask] = 0

                if random.random() < 0.4:
                    shift = random.randint(-5, 5)
                    img_array_aug = np.roll(img_array_aug, shift, axis=1)
                    if shift > 0:
                        img_array_aug[:, :shift] = 0
                    else:
                        img_array_aug[:, shift:] = 0

                if random.random() < 0.5:
                    img_array_aug = np.clip(img_array_aug * random.uniform(0.7, 1.3), 0, 255)

                img = Image.fromarray(img_array_aug.astype(np.uint8))

            label_dir = output_dir
            os.makedirs(label_dir, exist_ok=True)
            img.save(os.path.join(label_dir, f'{prefix}_{label}_{i}.png'))
            current_total += 1

    save_images_by_label(all_images, all_labels, output_dir=f'{folder}', prefix='all')

In [17]:
gen_image_into_a_folder('test/benign.csv', 'test/images/benign')
gen_image_into_a_folder('test/syn.csv', 'test/images/syn')
gen_image_into_a_folder('test/udp.csv', 'test/images/udp')

C:\Users\NewTun\AppData\Local\Temp\ipykernel_2236\268378244.py:45: RuntimeWarning: invalid value encountered in cast
  img_array = img_array.astype(np.uint8)
